# training-step-cycle — faded example 2: Training cycle with zero_grad blanked

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `training-step-cycle`. Running the beacon reports progress on the `PyTorch: Training step cycle` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Training step cycle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`training-step-cycle`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "training-step-cycle"
DD_SUBTOPIC = "PyTorch: Training step cycle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The `optimizer.zero_grad()` call clears the `.grad` attribute of every parameter the optimizer tracks. Without it, each call to `backward()` adds to existing gradients rather than replacing them. The result is that gradients grow unboundedly across steps, causing erratic or diverging updates.

## Faded exercise 2

Implement `faded2_train_zero_grad(w_init, x, y_target, lr, n_steps)`. Standard 5-call cycle on a scalar weight. The blank is the `optimizer.zero_grad()` call at the end of each iteration.

**Fill in:** The optimizer.zero_grad() call that clears accumulated gradients at the end of each training step.

In [ ]:
import torch as t

def faded2_train_zero_grad(w_init, x, y_target, lr, n_steps):
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y_target) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        raise NotImplementedError()  # TODO: The optimizer.zero_grad() call that clears accumulated gradients at the end of each training step.
    return w.detach().clone(), losses


def _test():
    import torch as t
    x = t.linspace(-1, 1, 10)
    y = 2.0 * x + 1.0
    w_final, losses = faded2_train_zero_grad(0.0, x, y, lr=0.05, n_steps=80)
    # w should be near 2.0
    assert abs(w_final.item() - 2.0) < 0.5, f'w = {w_final.item()}, expected ~2.0'
    assert losses[0] > losses[-1], 'loss did not decrease'
    # Compare to accumulating version to verify it differs
    # (no explicit check needed; above convergence check is sufficient)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def faded2_train_zero_grad(w_init, x, y_target, lr, n_steps):
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y_target) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    return w.detach().clone(), losses
```
</details>